In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/category_mappings.json
/kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/e-commerce_sequences.jsonl


In [2]:
import json
data = []

with open('/kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/e-commerce_sequences.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [3]:
for key in data[0]:
    print(key, len(data[0][key]))

user_id 8
slow_memory_ids 1
slow_id_timestamps 1
fast_memory_ids 15
fast_id_timestamps 15
target_item 7
target_timestamp 19
negative_items 100


In [4]:
with open('/kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/category_mappings.json', 'r') as f:
    mappings = json.load(f)

In [5]:
mappings.keys()

dict_keys(['product_to_category', 'category_to_code'])

In [6]:
mappings['product_to_category']['1003461']

'2053013555631882655'

In [7]:
mappings['category_to_code']['2053013555631882655']

'electronics.smartphone'

In [8]:
mappings['category_to_code'].values()

dict_values(['electronics.smartphone', 'appliances.kitchen.washer', 'computers.notebook', 'furniture.living_room.sofa', 'furniture.kitchen.chair', 'appliances.kitchen.dishwasher', 'electronics.audio.headphone', 'appliances.environment.water_heater', 'electronics.camera.video', 'electronics.clocks', 'electronics.video.tv', 'apparel.tshirt', 'apparel.jeans', 'construction.tools.drill', 'auto.accessories.compressor', 'computers.components.motherboard', 'computers.peripherals.printer', 'computers.desktop', 'appliances.kitchen.refrigerators', 'computers.components.videocards', 'furniture.bathroom.bath', 'furniture.bedroom.bed', 'electronics.audio.subwoofer', 'construction.tools.saw', 'electronics.clocks', 'furniture.kitchen.table', 'apparel.shoes', 'construction.tools.light', 'apparel.shirt', 'construction.tools.painting', 'apparel.shoes', 'furniture.kitchen.table', 'furniture.living_room.cabinet', 'auto.accessories.player', 'appliances.personal.massager', 'apparel.costume', 'computers.peri

In [9]:
import random
from collections import defaultdict
from tqdm import tqdm

category_to_products = defaultdict(list)
for product_id, cat_id in mappings['product_to_category'].items():
    category_to_products[cat_id].append(product_id)

NUM_NEGATIVES = 100

print("Replacing random negatives with HARD negatives")
for row in tqdm(data, desc="Sampling"):
    target_item = row['target_item']
    target_cat = mappings['product_to_category'].get(target_item)
    
    hard_negatives = []
    
    if target_cat and target_cat in category_to_products:
        same_cat_products = [p for p in category_to_products[target_cat] if p != target_item]
        
        if len(same_cat_products) >= NUM_NEGATIVES:
            hard_negatives = random.sample(same_cat_products, NUM_NEGATIVES)
        else:
            hard_negatives = same_cat_products
            
    # We fill the remaining slots with the standard random negatives already in the dataset
    if len(hard_negatives) < NUM_NEGATIVES:
        shortfall = NUM_NEGATIVES - len(hard_negatives)
        
        existing_negs = [n for n in row['negative_items'] if n not in hard_negatives and n != target_item]
        
        filler = existing_negs[:shortfall]
        hard_negatives.extend(filler)
        
    row['negative_items'] = hard_negatives

print(f"Sample row's new negatives length: {len(data[0]['negative_items'])}")
print("Done! The dataset now uses hard negatives.")

Replacing random negatives with HARD negatives


Sampling: 100%|██████████| 272074/272074 [00:26<00:00, 10141.94it/s]

Sample row's new negatives length: 100
Done! The dataset now uses hard negatives.


In [10]:
unique_products = set()
for d in data:
    unique_products.add(d['target_item'])
    for p_id in d['slow_memory_ids']:
        unique_products.add(p_id)
    for p_id in d['fast_memory_ids']:
        unique_products.add(p_id)
    for p_id in d['negative_items']:
        unique_products.add(p_id)

In [11]:
product_id_to_int = {}
current_int_id = 1

for string_id in unique_products:
    product_id_to_int[string_id] = current_int_id
    current_int_id += 1

num_unique_products = len(product_id_to_int)

In [12]:
data[0].keys()

dict_keys(['user_id', 'slow_memory_ids', 'slow_id_timestamps', 'fast_memory_ids', 'fast_id_timestamps', 'target_item', 'target_timestamp', 'negative_items'])

In [13]:
data[0]

{'user_id': '31198833',
 'slow_memory_ids': ['1005158'],
 'slow_id_timestamps': ['2019-11-08 02:09:45'],
 'fast_memory_ids': ['1003551',
  '1005158',
  '1004870',
  '1004873',
  '1004433',
  '1004436',
  '1002098',
  '1004785',
  '1004958',
  '1005253',
  '1005157',
  '1005158',
  '1004792',
  '1002101',
  '1003549'],
 'fast_id_timestamps': ['2019-11-08 02:10:32',
  '2019-11-08 02:10:53',
  '2019-11-08 02:14:16',
  '2019-11-08 02:15:58',
  '2019-11-08 02:20:01',
  '2019-11-08 02:20:53',
  '2019-11-08 02:24:52',
  '2019-11-08 02:28:29',
  '2019-11-08 02:30:03',
  '2019-11-08 02:32:06',
  '2019-11-08 02:33:10',
  '2019-11-08 02:35:02',
  '2019-11-08 02:38:46',
  '2019-11-08 02:39:48',
  '2019-11-08 02:40:28'],
 'target_item': '1005065',
 'target_timestamp': '2019-11-08 02:42:38',
 'negative_items': ['1004470',
  '1002629',
  '1004481',
  '1004376',
  '1003308',
  '1003461',
  '1004653',
  '1003615',
  '1005135',
  '1005083',
  '1005067',
  '1004792',
  '1005288',
  '1004292',
  '1004273'

In [14]:
unique_categories = set()
for d in data:
    unique_categories.add(mappings['product_to_category'][d['target_item']])
    for p_id in d['slow_memory_ids']:
        unique_categories.add(mappings['product_to_category'][p_id])
    for p_id in d['fast_memory_ids']:
        unique_categories.add(mappings['product_to_category'][p_id])
    for p_id in d['negative_items']:
        unique_categories.add(mappings['product_to_category'][p_id])

In [15]:

category_to_int = {}
current_cat_id = 1

for string_cat in unique_categories:
    category_to_int[string_cat] = current_cat_id
    current_cat_id += 1

num_unique_categories = len(category_to_int)
print(f"Total unique categories: {num_unique_categories}")

Total unique categories: 276


In [16]:
texts_to_encode = []
for cat in unique_categories:
    texts_to_encode.append(mappings['category_to_code'][cat])

In [17]:
texts_to_encode

['appliances.kitchen.oven',
 'construction.tools.painting',
 'appliances.kitchen.grill',
 'apparel.shoes',
 'appliances.sewing_machine',
 'appliances.environment.fan',
 'sport.bicycle',
 'apparel.glove',
 'medicine.tools.tonometer',
 'apparel.dress',
 'construction.tools.pump',
 'apparel.shoes.keds',
 'sport.bicycle',
 'furniture.bedroom.bed',
 'apparel.shoes.moccasins',
 'apparel.shoes.keds',
 'computers.components.power_supply',
 'appliances.kitchen.fryer',
 'apparel.shoes.sandals',
 'accessories.bag',
 'appliances.kitchen.kettle',
 'computers.components.videocards',
 'sport.snowboard',
 'appliances.kitchen.refrigerators',
 'sport.bicycle',
 'apparel.costume',
 'construction.tools.welding',
 'apparel.scarf',
 'apparel.shoes',
 'accessories.bag',
 'apparel.shoes',
 'electronics.smartphone',
 'apparel.shoes',
 'appliances.kitchen.steam_cooker',
 'appliances.personal.scales',
 'appliances.environment.vacuum',
 'furniture.kitchen.table',
 'sport.ski',
 'furniture.kitchen.table',
 'electr

In [18]:
import torch
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')
d_model = encoder.get_sentence_embedding_dimension()  # 384

num_unique_products = len(product_id_to_int)
pretrained_matrix = torch.zeros((num_unique_products + 1, d_model))

unique_cat_ids   = list(unique_categories)
cat_texts        = [mappings['category_to_code'][cat_id] for cat_id in unique_cat_ids]

print(f"Encoding {len(cat_texts)} category texts with SentenceTransformer...")
cat_embeddings = encoder.encode(
    cat_texts,
    batch_size=64,
    convert_to_tensor=True,
    show_progress_bar=True
)
cat_embeddings = cat_embeddings.cpu()  

cat_id_to_emb = {
    cat_id: cat_embeddings[i]
    for i, cat_id in enumerate(unique_cat_ids)
}

print(f"Mapping {num_unique_products} product rows...")
missing = 0
for product_str_id, int_id in product_id_to_int.items():
    cat_id = mappings['product_to_category'].get(product_str_id)
    if cat_id and cat_id in cat_id_to_emb:
        pretrained_matrix[int_id] = cat_id_to_emb[cat_id]
    else:
        missing += 1

print(f"\nMatrix shape          : {pretrained_matrix.shape}")
print(f"Row 0 (padding) norm  : {pretrained_matrix[0].norm().item():.4f}  (expected 0.0)")
print(f"Row 1 norm            : {pretrained_matrix[1].norm().item():.4f}  (expected > 0)")
print(f"Products with no mapping: {missing} / {num_unique_products}")

torch.save(pretrained_matrix, 'ecommerce_pretrained_matrix.pt')
print("\nPretrained matrix saved to 'ecommerce_pretrained_matrix.pt'.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 276 category texts with SentenceTransformer...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Mapping 87133 product rows...

Matrix shape          : torch.Size([87134, 384])
Row 0 (padding) norm  : 0.0000  (expected 0.0)
Row 1 norm            : 1.0000  (expected > 0)
Products with no mapping: 0 / 87133

Pretrained matrix saved to 'ecommerce_pretrained_matrix.pt'.


In [19]:
data[0]

{'user_id': '31198833',
 'slow_memory_ids': ['1005158'],
 'slow_id_timestamps': ['2019-11-08 02:09:45'],
 'fast_memory_ids': ['1003551',
  '1005158',
  '1004870',
  '1004873',
  '1004433',
  '1004436',
  '1002098',
  '1004785',
  '1004958',
  '1005253',
  '1005157',
  '1005158',
  '1004792',
  '1002101',
  '1003549'],
 'fast_id_timestamps': ['2019-11-08 02:10:32',
  '2019-11-08 02:10:53',
  '2019-11-08 02:14:16',
  '2019-11-08 02:15:58',
  '2019-11-08 02:20:01',
  '2019-11-08 02:20:53',
  '2019-11-08 02:24:52',
  '2019-11-08 02:28:29',
  '2019-11-08 02:30:03',
  '2019-11-08 02:32:06',
  '2019-11-08 02:33:10',
  '2019-11-08 02:35:02',
  '2019-11-08 02:38:46',
  '2019-11-08 02:39:48',
  '2019-11-08 02:40:28'],
 'target_item': '1005065',
 'target_timestamp': '2019-11-08 02:42:38',
 'negative_items': ['1004470',
  '1002629',
  '1004481',
  '1004376',
  '1003308',
  '1003461',
  '1004653',
  '1003615',
  '1005135',
  '1005083',
  '1005067',
  '1004792',
  '1005288',
  '1004292',
  '1004273'

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
from datetime import datetime

class RecSysDataset(Dataset):
    def __init__(self, data_list, product_id_to_int, category_to_int):
        self.data_list = data_list
        self.product_id_to_int = product_id_to_int
        self.category_to_int = category_to_int

    def __len__(self):
        return len(self.data_list)

    def _parse_time(self, time_str):
        return datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S')

    def __getitem__(self, idx):
        row = self.data_list[idx]

        raw_history_ids = row['slow_memory_ids'] + row['fast_memory_ids']
        raw_history_times = row['slow_id_timestamps'] + row['fast_id_timestamps']

        history_ids = [self.product_id_to_int[item_id] for item_id in raw_history_ids]
        history_cats = [self.category_to_int[mappings['product_to_category'][item_id]] for item_id in raw_history_ids]

        t_q = self._parse_time(row['target_timestamp'])
            
        history_delta_t = []
        for t_str in raw_history_times:
            interaction_time = self._parse_time(t_str)
            gap_hours = max(0.0, (t_q - interaction_time).total_seconds() / 3600.0)
            history_delta_t.append(gap_hours)

        candidate_strings = [row['target_item']] + row['negative_items']
        candidate_ids = [self.product_id_to_int[item_id] for item_id in candidate_strings]

        return {
            'history_seq': torch.tensor(history_ids, dtype=torch.long),
            'history_cat_seq': torch.tensor(history_cats, dtype=torch.long),
            'history_dt_seq': torch.tensor(history_delta_t, dtype=torch.float),
            'candidate_items': torch.tensor(candidate_ids, dtype=torch.long)
        }

In [21]:
from torch.nn.utils.rnn import pad_sequence

def recsys_collate_fn(batch):
    hist_seqs = [item['history_seq'] for item in batch]
    hist_cats = [item['history_cat_seq'] for item in batch]
    hist_dts = [item['history_dt_seq'] for item in batch]
    candidates = [item['candidate_items'] for item in batch]

    padded_hist_seqs = pad_sequence(hist_seqs, batch_first=True, padding_value=0)
    padded_hist_cats = pad_sequence(hist_cats, batch_first=True, padding_value=0)
    padded_hist_dts = pad_sequence(hist_dts, batch_first=True, padding_value=0.0)
    padded_candidates = pad_sequence(candidates, batch_first=True, padding_value=0)

    return {
        'history_seq': padded_hist_seqs,
        'history_cat_seq': padded_hist_cats,
        'history_dt_seq': padded_hist_dts,
        'candidate_items': padded_candidates
    }

In [22]:
len(data)

272074

In [23]:
from sklearn.model_selection import train_test_split

total_holdout = 20000

train_data, temp_data = train_test_split(
    data, 
    test_size=total_holdout, 
    random_state=42
)

val_data, test_data = train_test_split(
    temp_data, 
    test_size=0.5, 
    random_state=42
)

print(f"Data Split -> Train: {len(train_data)} | Valid: {len(val_data)} | Test: {len(test_data)}")

Data Split -> Train: 252074 | Valid: 10000 | Test: 10000


In [24]:
train_dataset = RecSysDataset(train_data, product_id_to_int, category_to_int)
val_dataset = RecSysDataset(val_data, product_id_to_int, category_to_int)
test_dataset = RecSysDataset(test_data, product_id_to_int, category_to_int)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=recsys_collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=recsys_collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=recsys_collate_fn, num_workers=2, pin_memory=True)

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CategoryAwareTemporalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.raw_gamma = nn.Parameter(torch.tensor(0.1))

    def forward(self, x, delta_t_prime, attn_mask=None):
        batch_size, seq_len, _ = x.size()
        
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        gamma = F.softplus(self.raw_gamma)
        penalty = gamma * torch.log1p(delta_t_prime) 
        penalty = penalty.view(batch_size, 1, 1, seq_len)
        
        logits = scores - penalty
        
        if attn_mask is not None:
            attn_mask = attn_mask.unsqueeze(1)
            logits = logits.masked_fill(~attn_mask, float('-inf'))
            
        attn_weights = F.softmax(logits, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.W_o(context)

import torch
import torch.nn as nn
import torch.nn.functional as F

class FinalRecModel(nn.Module):
    def __init__(self, num_items, num_categories, d_model, num_heads, category_dim=16, pretrained_matrix=None, max_fast_seq_len=20, cat_to_group=None, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_fast_seq_len = max_fast_seq_len
        if cat_to_group is None:
            raise ValueError("cat_to_group must be provided for category-wise personas")
        self.register_buffer("cat_to_group", cat_to_group.long())
        self.num_personas = int(self.cat_to_group.max().item() + 1)
        
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=d_model, padding_idx=0)
        if pretrained_matrix is not None:
            self.item_embedding.weight.data.copy_(pretrained_matrix)
        
        self.category_embedding = nn.Embedding(num_embeddings=num_categories, embedding_dim=category_dim, padding_idx=0)
        self.lambda_net = nn.Linear(category_dim, 1)

        self.pos_embedding = nn.Embedding(num_embeddings=max_fast_seq_len + self.num_personas, embedding_dim=d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

        self.attn_layer = CategoryAwareTemporalAttention(d_model, num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(d_model * 4, d_model)
        )
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _category_personas(self, all_embs, history_cat_seq, slow_mask):
        batch_size, seq_len, d_model = all_embs.size()
        personas = torch.zeros(batch_size, self.num_personas, d_model, device=all_embs.device)
        persona_mask = torch.zeros(batch_size, self.num_personas, dtype=torch.bool, device=all_embs.device)
        group_ids = self.cat_to_group[history_cat_seq]
        
        for g in range(self.num_personas):
            g_mask = slow_mask & (group_ids == g)
            if not g_mask.any():
                continue
            g_embs = all_embs * g_mask.unsqueeze(-1).float()
            g_counts = g_mask.sum(dim=1, keepdim=True).float()
            valid = g_counts.squeeze(1) > 0
            summed = g_embs.sum(dim=1)
            avg = summed / torch.clamp(g_counts, min=1.0)
            personas[valid, g] = avg[valid]
            persona_mask[valid, g] = True

        return personas, persona_mask

    def forward(self, history_seq, history_cat_seq, history_dt_seq, candidate_items=None):
        batch_size, seq_len = history_seq.size()
        device = history_seq.device

        # Compute Category-Aware Psychological Time
        c_hist = self.category_embedding(history_cat_seq)
        lambda_c = F.softplus(self.lambda_net(c_hist)).squeeze(-1)
        dt_prime = lambda_c * history_dt_seq 

        valid_mask = (history_seq != 0)
        dt_prime = dt_prime.masked_fill(~valid_mask, float('inf'))

        # Dynamic Eviction
        k = min(self.max_fast_seq_len, seq_len)
        fast_dt_prime, fast_indices = torch.topk(dt_prime, k=k, dim=1, largest=False)

        # Restore chronological order
        fast_indices, _ = torch.sort(fast_indices, dim=1) 
        fast_dt_prime = torch.gather(dt_prime, 1, fast_indices) 

        fast_valid_mask = torch.gather(valid_mask, 1, fast_indices)
        fast_dt_prime = fast_dt_prime.masked_fill(~fast_valid_mask, 0.0)

        # Gather Fast Memory embeddings
        fast_item_seq = torch.gather(history_seq, 1, fast_indices)
        x_fast = self.item_embedding(fast_item_seq)

        # Compile Slow Memory (Persona) via category-wise bins
        all_embs = self.item_embedding(history_seq)
        fast_mask = torch.zeros_like(history_seq, dtype=torch.bool).scatter_(1, fast_indices, True)
        slow_mask = valid_mask & (~fast_mask)
        personas, persona_mask = self._category_personas(all_embs, history_cat_seq, slow_mask)

        # Combine Persona Prefixes with Fast Sequence
        x_combined = torch.cat([personas, x_fast], dim=1) 
        
        delta_t_0 = torch.zeros(batch_size, self.num_personas, device=device)
        delta_t_combined = torch.cat([delta_t_0, fast_dt_prime], dim=1) 

        # Positional Embeddings
        total_seq_len = k + self.num_personas
        positions = torch.arange(total_seq_len, dtype=torch.long, device=device).unsqueeze(0).expand(batch_size, total_seq_len)
        x_combined = x_combined + self.pos_embedding(positions)
        x_combined = self.dropout(x_combined)

        causal_mask = torch.tril(torch.ones(total_seq_len, total_seq_len, device=device)).bool()
        combined_valid_mask = torch.cat([persona_mask, fast_valid_mask], dim=1)
        
        full_attn_mask = causal_mask.unsqueeze(0) & combined_valid_mask.unsqueeze(1)

        attn_out = self.attn_layer(x_combined, delta_t_combined, full_attn_mask)
        x = self.layer_norm1(x_combined + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        h_out = self.layer_norm2(x + self.dropout(ffn_out))

        last_valid_idx = combined_valid_mask.sum(dim=1) - 1
        batch_indices = torch.arange(batch_size, device=device)
        h_final = h_out[batch_indices, last_valid_idx, :]
        if candidate_items is not None:
            candidate_embs = self.item_embedding(candidate_items)
            h_final_norm = F.normalize(h_final, p=2, dim=-1).unsqueeze(1)
            candidate_embs_norm = F.normalize(candidate_embs, p=2, dim=-1).transpose(1, 2)
            
            prediction_logits = torch.bmm(h_final_norm, candidate_embs_norm).squeeze(1)
            prediction_logits = prediction_logits * 10.0 
        else:
            h_final_norm = F.normalize(h_final, p=2, dim=-1)
            item_embs_norm = F.normalize(self.item_embedding.weight, p=2, dim=-1)
            prediction_logits = torch.matmul(h_final_norm, item_embs_norm.T) * 10.0

        return prediction_logits

In [26]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
import math

LEARNING_RATE = 5e-4     
WEIGHT_DECAY = 1e-4        
EPOCHS = 30        
NUM_HEADS = 4              

def calculate_hr_at_k(logits, targets, k=10):
    _, top_indices = torch.topk(logits, k=k, dim=1)
    
    targets_expanded = targets.unsqueeze(1).expand_as(top_indices)
    hits = (top_indices == targets_expanded).sum().item()
    
    return hits

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(f"Training on device: {device}")

num_items = pretrained_matrix.size(0) 
d_model = pretrained_matrix.size(1)


total_category_slots = len(category_to_int) + 1
top_groups = {}
for cat_id_str, cat_int in category_to_int.items():
    full_name = mappings['category_to_code'].get(cat_id_str, "unknown")
    top_name = full_name.split('.')[0] if full_name else "unknown"
    if top_name not in top_groups:
        top_groups[top_name] = len(top_groups)
cat_to_group = torch.zeros(total_category_slots, dtype=torch.long)
for cat_id_str, cat_int in category_to_int.items():
    full_name = mappings['category_to_code'].get(cat_id_str, "unknown")
    top_name = full_name.split('.')[0] if full_name else "unknown"
    cat_to_group[cat_int] = top_groups[top_name]
num_personas = len(top_groups)
print(f"Top-level category personas: {num_personas}")

model = FinalRecModel(
    num_items=num_items, 
    num_categories=total_category_slots,
    d_model=d_model, 
    num_heads=NUM_HEADS, 
    pretrained_matrix=pretrained_matrix,
    max_fast_seq_len=20,   
    cat_to_group=cat_to_group,
    dropout_rate=0.1
).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_loss = float('inf')


for epoch in range(1, EPOCHS + 1):
    
    model.train()
    total_train_loss = 0.0
    
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False)
    
    for batch in train_progress:
        history_seq = batch['history_seq'].to(device)
        history_cat_seq = batch['history_cat_seq'].to(device)
        history_dt_seq = batch['history_dt_seq'].to(device)

        candidate_items = batch['candidate_items'].to(device)
        
        batch_size = history_seq.size(0)
        target_labels = torch.zeros(batch_size, dtype=torch.long, device=device)
        
        optimizer.zero_grad()
        
        logits = model(history_seq, history_cat_seq, history_dt_seq, candidate_items)
        loss = F.cross_entropy(logits, target_labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        train_progress.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_val_loss = 0.0

    total_hits_10 = 0
    total_samples = 0
    
    val_progress = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Valid]", leave=False)
    
    with torch.no_grad():
        for batch in val_progress:
            history_seq = batch['history_seq'].to(device)
            history_cat_seq = batch['history_cat_seq'].to(device)
            history_dt_seq = batch['history_dt_seq'].to(device)
            candidate_items = batch['candidate_items'].to(device)
            
            batch_size = history_seq.size(0)
            
            target_labels = torch.zeros(batch_size, dtype=torch.long, device=device)
            
            logits = model(history_seq, history_cat_seq, history_dt_seq, candidate_items)
            loss = F.cross_entropy(logits, target_labels)
            
            total_val_loss += loss.item()
            
            hits = calculate_hr_at_k(logits, target_labels, k=10)
            total_hits_10 += hits
            total_samples += batch_size
            
            val_progress.set_postfix({
                'val_loss': f"{loss.item():.4f}", 
                'HR@10': f"{(total_hits_10/total_samples):.4f}" # Update live
            })
            
    avg_val_loss = total_val_loss / len(val_loader)
    hr_at_10 = total_hits_10 / total_samples
    
    print(f"Epoch {epoch}/{EPOCHS} | Valid Loss: {avg_val_loss:.4f} | Valid HR@10: {hr_at_10:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_final_model.pth")
        print("  -> Validation loss improved. Model saved!")
    print("-" * 60)

print("Training Complete! Best model saved to 'best_final_model.pth'.")

Training on device: cuda
Top-level category personas: 13


Epoch 1/30 | Valid Loss: 3.2301 | Valid HR@10: 0.6581
  -> Validation loss improved. Model saved!
------------------------------------------------------------


Epoch 2/30 | Valid Loss: 3.1338 | Valid HR@10: 0.6874
  -> Validation loss improved. Model saved!
------------------------------------------------------------


Epoch 3/30 | Valid Loss: 3.0903 | Valid HR@10: 0.6980
  -> Validation loss improved. Model saved!
------------------------------------------------------------


Epoch 4/30 | Valid Loss: 3.0151 | Valid HR@10: 0.7189
  -> Validation loss improved. Model saved!
------------------------------------------------------------


...

Epoch 19/30 | Valid Loss: 2.5688 | Valid HR@10: 0.7942
  -> Validation loss improved. Model saved!
------------------------------------------------------------


Epoch 20/30 | Valid Loss: 2.5663 | Valid HR@10: 0.7930
  -> Validation loss improved. Model saved!
------------------------------------------------------------


Epoch 21/30 | Valid Loss: 2.5480 | Valid HR@10: 0.7972
  -> Validation loss improved. Model saved!
------------------------------------------------------------


Epoch 22/30 | Valid Loss: 2.5527 | Valid HR@10: 0.7931
------------------------------------------------------------


KeyboardInterrupt: 

In [27]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm


def get_ranks(ps, ns):
    all_s = torch.cat([ps.unsqueeze(1), ns], dim=1)
    ranked = torch.argsort(all_s, dim=1, descending=True)
    pos_rank = (ranked == 0).nonzero(as_tuple=True)[1]
    return pos_rank + 1, all_s 

def compute_metrics(ps, ns, ks=[5, 10]):
    ranks, all_s = get_ranks(ps, ns)
    ranks_float = ranks.float()
    
    # Mean MRR
    mrr = (1.0 / ranks_float).mean().item()
    
    # NDCG@K
    metrics = {f'ndcg@{k}': 0 for k in ks}
    for k in ks:
        # NDCG = 1/log2(rank + 1) if rank <= k else 0
        ndcg_mask = (ranks <= k).float()
        ndcg = (ndcg_mask / torch.log2(ranks_float + 1)).mean().item()
        metrics[f'ndcg@{k}'] = ndcg
    
    # Group AUC
    num_neg = ns.size(1)
    gauc = ((num_neg + 1 - ranks_float) / num_neg).mean().item()
    
    metrics['mrr'] = mrr
    metrics['gauc'] = gauc
    return metrics

def hr_at_k(ps, ns, k=10):
    all_s = torch.cat([ps.unsqueeze(1), ns], dim=1)
    ranked = torch.argsort(all_s, dim=1, descending=True)
    pos_rank = (ranked == 0).nonzero(as_tuple=True)[1] # 0-indexed rank
    return (pos_rank < k).float().mean().item()



# Load the best model weights
model.load_state_dict(torch.load("best_final_model.pth", weights_only=True))
model.eval()
print("Successfully loaded 'best_final_model.pth'. Starting evaluation...")

test_metrics = {
    'loss': 0.0, 
    'gauc': 0.0, 
    'mrr': 0.0, 
    'hr@10': 0.0, 
    'ndcg@5': 0.0, 
    'ndcg@10': 0.0
}
nb_batches = 0

val_progress = tqdm(val_loader, desc="Validating")

with torch.no_grad():
    for batch in val_progress:
        # Move inputs to device
        history_seq = batch['history_seq'].to(device)
        history_cat_seq = batch['history_cat_seq'].to(device)
        history_dt_seq = batch['history_dt_seq'].to(device)
        candidate_items = batch['candidate_items'].to(device)
        
        batch_size = history_seq.size(0)
        
        # Forward pass
        logits = model(history_seq, history_cat_seq, history_dt_seq, candidate_items)
        
        # Calculate CE loss (since this is what determined the "best_loss")
        target_labels = torch.zeros(batch_size, dtype=torch.long, device=device)
        loss = F.cross_entropy(logits, target_labels)
        
        # Split logits into positive and negative scores
        ps = logits[:, 0]   # Score of the actual target item
        ns = logits[:, 1:]  # Scores of the 100 negative items
        
        # Calculate all metrics
        batch_metrics = compute_metrics(ps, ns, ks=[5, 10])
        batch_hr10 = hr_at_k(ps, ns, k=10)
        
        # Accumulate
        test_metrics['loss'] += loss.item()
        test_metrics['gauc'] += batch_metrics['gauc']
        test_metrics['mrr'] += batch_metrics['mrr']
        test_metrics['hr@10'] += batch_hr10
        test_metrics['ndcg@5'] += batch_metrics['ndcg@5']
        test_metrics['ndcg@10'] += batch_metrics['ndcg@10']
        
        nb_batches += 1


# Average out the metrics
for key in test_metrics.keys():
    test_metrics[key] /= nb_batches

print("\n" + "="*40)
print(" Validation Set Performance ")
print("="*40)
for key, val in test_metrics.items():
    print(f"{key.upper():<10} : {val:.4f}")
print("="*40)

Successfully loaded 'best_final_model.pth'. Starting evaluation...


Validating: 100%|██████████| 79/79 [00:22<00:00,  3.47it/s]


 Validation Set Performance 
LOSS       : 2.5480
GAUC       : 0.9288
MRR        : 0.5096
HR@10      : 0.7988
NDCG@5     : 0.5338
NDCG@10    : 0.5715


In [30]:
import torch
import torch.nn.functional as F

int_to_category_name = {}
for cat_id_str, cat_int in category_to_int.items():
    readable_name = mappings['category_to_code'].get(cat_id_str, "Unknown")
    int_to_category_name[cat_int] = readable_name

model.eval()
decay_rates = []

with torch.no_grad():
    for cat_int in range(1, len(category_to_int) + 1):
        cat_tensor = torch.tensor([cat_int], dtype=torch.long, device=device)
        
        c_emb = model.category_embedding(cat_tensor)
        lambda_val = F.softplus(model.lambda_net(c_emb)).item()
        
        name = int_to_category_name[cat_int]
        decay_rates.append((name, lambda_val))

decay_rates.sort(key=lambda x: x[1], reverse=True)

for name, rate in decay_rates:
    print(f"{name:<45} : {rate:.4f}")


auto.accessories.alarm                        : 2.1501
accessories.bag                               : 1.7778
construction.tools.light                      : 1.6445
kids.skates                                   : 1.6239
furniture.living_room.cabinet                 : 1.6131
apparel.trousers                              : 1.5761
sport.bicycle                                 : 1.5362
accessories.bag                               : 1.5123
country_yard.cultivator                       : 1.5003
electronics.audio.headphone                   : 1.4487
apparel.costume                               : 1.4102
apparel.jeans                                 : 1.3645
electronics.video.tv                          : 1.3623
construction.tools.welding                    : 1.3519
apparel.shoes                                 : 1.3496
apparel.costume                               : 1.3469
furniture.living_room.cabinet                 : 1.3442
furniture.bathroom.toilet                     : 1.3412
auto.acces